# End-to-end pipeline (Gradio on Kaggle)

Input: **one PDF** or **directory of PDFs**.  
Output: **one PDF** or **directory of PDFs** with summary merged after metadata.

**Run order:** 1) Install dependencies (below, once per session; restart kernel after). 2) Run pipeline stages if needed each stage files 3) Use Gradio UI when to run full pipeline.


## 1. Install dependencies

In [ ]:
!pip uninstall -y transformers torchvision torch
!pip install torch==2.9.0 torchvision
!pip install transformers==4.57.1
!pip install surya-ocr

In [ ]:
# PDF extraction
!pip install pdfplumber pdf2image pdfminer.six
!apt-get update && apt-get install -y poppler-utils

# other dependencies
!pip install peft accelerate safetensors "huggingface_hub>=0.28.0"
!pip install regex networkx scikit-learn numpy scipy tqdm PyMuPDF pytesseract pandas matplotlib
!pip install torch-geometric

## Restart kernel


## 2. Import strategy

**Import each stage when we run it** Reasons:

- **Memory:** Surya, NLLB, BERT, GNN, BART are large. Loading them one stage at a time avoids OOM.
- **Clarity:** If a stage fails, later stages are not imported yet, so errors are easier to trace.


## 3. Workspace and input

Set a single workspace directory; each stage writes to a subdir.  
**Input** can be one PDF file or a directory of PDFs.

In [ ]:
import sys
from pathlib import Path
import io, contextlib
import zipfile
import shutil
import gradio as gr
import logging
import time
for _ in ("uvicorn", "uvicorn.error", "starlette"):
    logging.getLogger(_).setLevel(logging.CRITICAL)

# Pipeline root
PIPELINE_ROOT = Path("/kaggle/input/datasets/muaadhnazly/full-pipeline")
if not PIPELINE_ROOT.exists():
    PIPELINE_ROOT = Path("/kaggle/working/full-pipeline")
if not PIPELINE_ROOT.exists():
    PIPELINE_ROOT = Path.cwd()
sys.path.insert(0, str(PIPELINE_ROOT))

WORKSPACE = Path("/kaggle/working/pipeline_workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Input: one PDF file OR a directory of PDFs.
INPUT_PDF_OR_DIR = Path("/kaggle/input/datasets/muaadhnazly/new-files/SC_CA_04_2022.pdf")

# Stage outputs
EXTRACTED_DIR = WORKSPACE / "01_extracted"
CLEANED_DIR = WORKSPACE / "02_cleaned"
TRANSLATED_DIR = WORKSPACE / "03_translated"
METASPLIT_DIR = WORKSPACE / "04_metasplit"
BODY_JOINED_DIR = WORKSPACE / "05_body_joined"
SENTENCES_DIR = WORKSPACE / "06_sentences"
CLAUSES_DIR = WORKSPACE / "07_clauses"
CLAUSE_JSON_DIR = WORKSPACE / "08_clause_jsons"
PREDICTED_DIR = WORKSPACE / "09_predicted"
GRAPHS_DIR = WORKSPACE / "10_graphs"
RANKED_DIR = WORKSPACE / "11_ranked"
ABSTRACTED_DIR = WORKSPACE / "12_abstracted"
FINAL_PDF_DIR = WORKSPACE / "13_final_pdf"
OCR_TMP_DIR = WORKSPACE / "ocr_tmp"
for d in (EXTRACTED_DIR, CLEANED_DIR, TRANSLATED_DIR, METASPLIT_DIR, BODY_JOINED_DIR, SENTENCES_DIR, CLAUSES_DIR, CLAUSE_JSON_DIR, PREDICTED_DIR, GRAPHS_DIR, RANKED_DIR, ABSTRACTED_DIR, FINAL_PDF_DIR, OCR_TMP_DIR):
    d.mkdir(parents=True, exist_ok=True)

## 4. Model paths

Set paths to Kaggle datasets for NLLB, InCaseLawBERT, and GNN.

In [ ]:
import sys
from pathlib import Path

if "PIPELINE_ROOT" in dir() and str(PIPELINE_ROOT) not in sys.path:
    sys.path.insert(0, str(PIPELINE_ROOT))
import pipeline_config


# NLLB LoRA adapter
pipeline_config.NLLB_PATH = Path("/kaggle/input/datasets/muaadhnazly/fine-tuned-1-3b/nllb_sinhala2english_lora")

# InCaseLawBERT
pipeline_config.INCASELAWBERT_PATH = Path("/kaggle/input/datasets/muaadhnazly/incaselawbert/final_model")

# GNN .pt
pipeline_config.GNN_MODEL_PATH = Path("/kaggle/input/datasets/muaadhnazly/gnn-trained/rgcn_encoder_final.pt")

### Stage 1: PDF extraction

Import runs Surya model load once. Single PDF → one `.txt` in `01_extracted`; directory → one `.txt` per PDF, preserving relative structure.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    from preprocess.pdf_extraction import process_pdf, process_all_pdfs
    if INPUT_PDF_OR_DIR.is_file():
        out_txt = EXTRACTED_DIR / (INPUT_PDF_OR_DIR.stem + ".txt")
        process_pdf(INPUT_PDF_OR_DIR, out_txt, OCR_TMP_DIR)
    else:
        process_all_pdfs(INPUT_PDF_OR_DIR, EXTRACTED_DIR, OCR_TMP_DIR)
print("Extracted")

### Stage 2: Clean text

Reads `.txt` from `01_extracted`, writes `.cleaned.txt` to `02_cleaned`.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    from preprocess.clean_text import process_directory_recursive
    process_directory_recursive(str(EXTRACTED_DIR), str(CLEANED_DIR))
print("Cleaned")

### Stage 3: Translate (NLLB + LoRA)

Reads `.cleaned.txt` from `02_cleaned`, writes `.translated.txt` to `03_translated`. Adapter loaded from `preprocess/nllb_sinhala2english_lora`.

In [ ]:
_nllb = getattr(__import__("pipeline_config", fromlist=["NLLB_PATH"]), "NLLB_PATH", None)
_nllb_str = str(_nllb) if _nllb else None
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    from preprocess.translator import process_directory
    process_directory(str(CLEANED_DIR), str(TRANSLATED_DIR), adapter_dir=_nllb_str)

print("Translated")

### Stage 4: Meta split + body join

Split METADATA/BODY from `03_translated`, then join all lines after `=== BODY ===` into one line for sentence splitting. Output in `05_body_joined`.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    from preprocess.meta_splitter import process_dir, BodyLineJoiner
    process_dir(TRANSLATED_DIR, METASPLIT_DIR)
    BodyLineJoiner().process_directory(str(METASPLIT_DIR), str(BODY_JOINED_DIR))
print("Meta split")

### Stage 5: Sentence split

Reads body-joined `.txt` from `05_body_joined`, writes `.sentences.txt` to `06_sentences`. Keeps METADATA/BODY structure.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    from preprocess.sentence_splitter import process_directory
    process_directory(str(BODY_JOINED_DIR), str(SENTENCES_DIR))
print("Sentence split")

### Stage 6: Clause split

Reads `.sentences.txt` from `06_sentences`, writes `.clauses.txt` (one clause per line) to `07_clauses`.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()):
    from preprocess.clause_splitter import process_all_files
    process_all_files(SENTENCES_DIR, CLAUSES_DIR)
print("Clause split")

### Stage 7: Clauses → JSON

Convert each `.clauses.txt` in `07_clauses` to one JSON per doc with `doc_id` and `clauses` (each clause has `text`, `prev_text`, `next_text`, `prev_text_1`). Output in `08_clause_jsons`.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from tag_clause.clauses_to_json import clauses_dir_to_json_dir
    n = clauses_dir_to_json_dir(CLAUSES_DIR, CLAUSE_JSON_DIR)
print(f"Clauses → JSON ({n} docs)")

### Stage 8: Predict clause labels (InCaseLawBERT)

Load InCaseLawBERT from `tag_clause/InCaseLawBERT`, predict Premise/None/Opposition/Claim for each clause. Reads `08_clause_jsons`, writes one JSON per doc to `09_predicted` with `label` per clause.

In [ ]:
_bert = getattr(__import__("pipeline_config", fromlist=["INCASELAWBERT_PATH"]), "INCASELAWBERT_PATH", None)
_model_path = str(_bert) if _bert else str(PIPELINE_ROOT / "tag_clause" / "InCaseLawBERT")
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    from tag_clause.predict_clauses import predict
    predict(model_path=_model_path, input_dir=str(CLAUSE_JSON_DIR), output_dir=str(PREDICTED_DIR), batch_size=16)

print("Clauses predicted")

### Stage 9: Graph construction

Build base graphs from `09_predicted` JSON (clause labels), add semantic edges (InCaseLawBert),  Writes  JSONs to `10_graphs` (e.g. `semantic_graphs/*_semantic.json`).

In [ ]:
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    from graph.construct_graph import run_construct_graph
    run_construct_graph(str(PREDICTED_DIR),
                        str(GRAPHS_DIR),
                        citations_path=str(PIPELINE_ROOT / "graph" / "graph_citations.json"),
                        bert_model_path=_model_path)


print("Graph built")

### Stage 10: Rank clauses

Rank clauses using graph centrality, citation importance, and GNN diversity. Reads `10_graphs/semantic_graphs/*_semantic.json`, writes `11_ranked/*.clauses_ranked.json`.

In [ ]:
_gnn = getattr(__import__("pipeline_config", fromlist=["GNN_MODEL_PATH"]), "GNN_MODEL_PATH", None)
_gnn_str = str(_gnn) if _gnn else str(PIPELINE_ROOT / "graph" / "rgcn_encoder.pt")
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    from ranking.rank_clauses import process_all_graphs
    import torch
    process_all_graphs(graph_dir=str(GRAPHS_DIR), output_dir=str(RANKED_DIR), gnn_model_path=_gnn_str, bert_model_path=_model_path, use_diversity=True, adaptive_topk=True, device="cuda" if torch.cuda.is_available() else "cpu")

print("Ranked")

### Stage 11: Abstractive summarization

BART-large-CNN summarizes selected clauses; writes `12_abstracted/*.abstracted.json` with `summary_final` and bullets. Reads `11_ranked/*.clauses_ranked.json`.

In [ ]:
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from abstraction_merge.abstraction import run_abstraction
    run_abstraction(str(RANKED_DIR), str(ABSTRACTED_DIR))
print("Abstracted")

### Stage 12: Merge summary into PDF

Finds metadata boundary in each PDF, appends summary from `12_abstracted/*.abstracted.json`, writes `13_final_pdf/*_summarized.pdf`.

In [ ]:
pdf_dir = INPUT_PDF_OR_DIR if INPUT_PDF_OR_DIR.is_dir() else INPUT_PDF_OR_DIR.parent
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    from abstraction_merge.merge import process_supreme_court_dir
    process_supreme_court_dir(pdf_dir=str(pdf_dir), summary_dir=str(ABSTRACTED_DIR), out_dir=str(FINAL_PDF_DIR))
print("Merged")

---
## Gradio UI: Upload PDF(s), run pipeline, download results

Choose Single PDF or Directory of PDFs, upload file(s), then click Run pipeline. Log appears as each stage finishes and finally zip can be download as results.

In [ ]:
INPUT_DIR_FOR_UI = WORKSPACE / "input"
pipeline_run_finished = False

def run_pipeline(input_type: str, files):
    """input_type: 'Single PDF' or 'Directory of PDFs'. files: list of uploaded paths."""
    log = []
    if files is None:
        files = []
    if isinstance(files, str):
        files = [files]
    elif not isinstance(files, list):
        files = list(files) if files else []
    if not files:
        yield "Please upload at least one PDF.", None, gr.update(interactive=True)
        return
    if input_type == "Single PDF" and len(files) != 1:
        yield "For 'Single PDF', upload exactly one file.", None, gr.update(interactive=True)
        return

    # Copy uploads to workspace/input
    INPUT_DIR_FOR_UI.mkdir(parents=True, exist_ok=True)
    for f in INPUT_DIR_FOR_UI.iterdir():
        f.unlink()
    paths = [p if isinstance(p, str) else getattr(p, "name", None) or str(p) for p in files]
    for src in paths:
        if not src or not Path(src).exists():
            continue
        shutil.copy2(src, INPUT_DIR_FOR_UI / Path(src).name)
    pdfs = list(INPUT_DIR_FOR_UI.glob("*.pdf"))
    if not pdfs:
        yield "No PDF files found in upload.", None, gr.update(interactive=True)
        return
    if input_type == "Single PDF":
        input_path = pdfs[0]
    else:
        input_path = INPUT_DIR_FOR_UI

    global INPUT_PDF_OR_DIR
    INPUT_PDF_OR_DIR = input_path
    yield "Starting pipeline…", None, gr.update(interactive=False)
    try:
        # Extraction
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from preprocess.pdf_extraction import process_pdf, process_all_pdfs
            if INPUT_PDF_OR_DIR.is_file():
                out_txt = EXTRACTED_DIR / (INPUT_PDF_OR_DIR.stem + ".extracted.txt")
                process_pdf(INPUT_PDF_OR_DIR, out_txt, OCR_TMP_DIR)
            else:
                process_all_pdfs(INPUT_PDF_OR_DIR, EXTRACTED_DIR, OCR_TMP_DIR)
        log.append("Extracted")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Cleaning
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from preprocess.clean_text import process_directory_recursive
            process_directory_recursive(str(EXTRACTED_DIR), str(CLEANED_DIR))
        log.append("Cleaned")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Translation
        _nllb = getattr(__import__("pipeline_config", fromlist=["NLLB_PATH"]), "NLLB_PATH", None)
        _nllb_str = str(_nllb) if _nllb else None
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from preprocess.translator import process_directory
            process_directory(str(CLEANED_DIR), str(TRANSLATED_DIR), adapter_dir=_nllb_str)
        log.append("Translated")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Meta split + body join
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from preprocess.meta_splitter import process_dir, BodyLineJoiner
            process_dir(TRANSLATED_DIR, METASPLIT_DIR)
            BodyLineJoiner().process_directory(str(METASPLIT_DIR), str(BODY_JOINED_DIR))
        log.append("Meta split")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Sentence split
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from preprocess.sentence_splitter import process_directory as process_sentences
            process_sentences(str(BODY_JOINED_DIR), str(SENTENCES_DIR))
        log.append("Sentence split")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Clause split
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from preprocess.clause_splitter import process_all_files
            process_all_files(SENTENCES_DIR, CLAUSES_DIR)
        log.append("Clause split")
        yield "\n".join(log), None, gr.update(interactive=False)
        log.append("")

        # Clauses → JSON
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from tag_clause.clauses_to_json import clauses_dir_to_json_dir
            clauses_dir_to_json_dir(CLAUSES_DIR, CLAUSE_JSON_DIR)
        log.append("Clauses → JSON")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Predict clause labels
        _bert = getattr(__import__("pipeline_config", fromlist=["INCASELAWBERT_PATH"]), "INCASELAWBERT_PATH", None)
        _model_path = str(_bert) if _bert else str(PIPELINE_ROOT / "tag_clause" / "InCaseLawBERT")
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from tag_clause.predict_clauses import predict
            predict(model_path=_model_path, input_dir=str(CLAUSE_JSON_DIR), output_dir=str(PREDICTED_DIR), batch_size=16)
        log.append("Predict clauses")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Graph construction
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from graph.construct_graph import run_construct_graph
            run_construct_graph(str(PREDICTED_DIR), str(GRAPHS_DIR), citations_path=str(PIPELINE_ROOT / "graph" / "graph_citations.json"), bert_model_path=_model_path)
        log.append("Graph built")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Rank clauses
        _gnn = getattr(__import__("pipeline_config", fromlist=["GNN_MODEL_PATH"]), "GNN_MODEL_PATH", None)
        _gnn_str = str(_gnn) if _gnn else str(PIPELINE_ROOT / "graph" / "rgcn_encoder.pt")
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from ranking.rank_clauses import process_all_graphs
            import torch
            process_all_graphs(graph_dir=str(GRAPHS_DIR), output_dir=str(RANKED_DIR), gnn_model_path=_gnn_str, bert_model_path=_model_path, use_diversity=True, adaptive_topk=True, device="cuda" if torch.cuda.is_available() else "cpu")
        log.append("Ranked")
        yield "\n".join(log), None, gr.update(interactive=False)

        # Abstractive summarization
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from abstraction_merge.abstraction import run_abstraction
            run_abstraction(str(RANKED_DIR), str(ABSTRACTED_DIR))
        log.append("Abstracted")
        yield "\n".join(log), None, gr.update(interactive=False)
        pdf_dir = INPUT_PDF_OR_DIR if INPUT_PDF_OR_DIR.is_dir() else INPUT_PDF_OR_DIR.parent

        # Merge summary into PDF
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            from abstraction_merge.merge import process_supreme_court_dir
            process_supreme_court_dir(pdf_dir=str(pdf_dir), summary_dir=str(ABSTRACTED_DIR), out_dir=str(FINAL_PDF_DIR))
        log.append("Merged")
        yield "\n".join(log), None, gr.update(interactive=False)
        log.append("")
        log.append("Done. Outputs in 01_extracted → 13_final_pdf under pipeline_workspace.")

        # Zip outputs
        zip_path = None
        if FINAL_PDF_DIR.is_dir():
            pdf_out = list(FINAL_PDF_DIR.glob("*.pdf"))
            if pdf_out:
                zip_path = WORKSPACE / "pipeline_outputs.zip"
                with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
                    for f in pdf_out:
                        z.write(f, f.name)
                zip_path = str(zip_path)
        global pipeline_run_finished
        pipeline_run_finished = True
        yield "\n".join(log), zip_path, gr.update(interactive=True)
    except Exception as e:
        log.append(f"Error: {e}")
        import traceback
        log.append(traceback.format_exc())
        yield "\n".join(log), None, gr.update(interactive=True)

with gr.Blocks(title="Pipeline (Stages 1–6)") as demo:
    gr.Markdown("### Run pipeline")
    input_type = gr.Radio(
        choices=["Single PDF", "Directory of PDFs"],
        value="Single PDF",
        label="Input type"
    )
    file_in = gr.File(
        label="Upload PDF(s) (one for single, multiple for directory)",
        file_count="multiple",
        file_types=[".pdf"]
    )
    run_btn = gr.Button("Run pipeline (Stages 1–12)")
    log_out = gr.Textbox(label="Log", lines=12, interactive=False)
    download_out = gr.File(label="Download results (zip)")
    run_btn.click(
        fn=run_pipeline,
        inputs=[input_type, file_in],
        outputs=[log_out, download_out, run_btn]
    )

demo.launch()

print("Gradio running. Run the pipeline from the UI; cell will stop after one run completes.")
while not pipeline_run_finished:
    time.sleep(10)
print("Pipeline run finished. Cell stopped.")